# Notebook 2 — LID & Mahalanobis Feature Extraction

**Pipeline position:** stage 2 of 5. It turns the raw `{clean, adv, noisy}` images from
Notebook 1 into the two **unsupervised score features** that the ensemble later stacks:

- **Mahalanobis** confidence scores — per layer, with input
  pre-processing, over a sweep of noise magnitudes `m`.
- **LID** (Local Intrinsic Dimensionality) — per layer, over a sweep of
  neighbourhood sizes `k` (`overlap`).

For each dataset/attack it writes one `.npy` per hyper-parameter value. Every file has the
same layout the rest of the pipeline expects:

```
rows  = [ adversarial | clean | noisy ]      (each truncated to a multiple of 100)
cols  = [ score_layer0 ... score_layer4 | label ]   label: adv=0, clean/noisy=1
```

Notebook 3 then does model selection over `m` and `k` and keeps the single best file each.

### Attribution
The scoring methods here are **prior work**, re-implemented on modern PyTorch:

- **Mahalanobis detector** (`sample_estimator`, input-preprocessed layer scores) — Lee et al.,
  *A Simple Unified Framework...* (NeurIPS 2018), repo `pokaxpoka/deep_Mahalanobis_detector`.
- **LID estimator** (`mle_batch`) — Ma et al., *Characterizing Adversarial Subspaces Using
  Local Intrinsic Dimensionality* (ICLR 2018).

The contribution of this project lives in later notebooks (The design of supervised detectors and GA-optimized ensemble implementation) — not in attack generation itself.

### Requirements to run on Kaggle
- **GPU T4** (Settings → Accelerator → GPU **T4**).
- **Internet ON** (for the `git clone`).
- **Add data:** the `attacked-pth-files` dataset from Notebook 1, and the pretrained
  weights dataset (adjust `weights_dir` below to your path).

In [ ]:
import os, sys, subprocess, warnings
warnings.filterwarnings("ignore")

REPO = "deep_Mahalanobis_detector"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "--quiet",
         "https://github.com/pokaxpoka/deep_Mahalanobis_detector.git"],
        check=True,
    )
sys.path.append("./" + REPO)

import numpy as np
import torch
import torch.nn.functional as F
import torchvision.transforms as T
import sklearn.covariance
from scipy.spatial.distance import cdist
from sklearn.metrics import roc_auc_score

from deep_Mahalanobis_detector import models, data_loader  

SEED = 0
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE, "| torch", torch.__version__)

Device: cuda | torch 2.10.0+cu128


In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Config:
    ds_name: str  = "cifar10"          # 'cifar10' | 'svhn' 
    net_type: str = "resnet"
    adv_type: str = "FGSM"             # 'FGSM' | 'BIM' | 'DeepFool' | 'CWL2'
    batch_size: int = 100              
    attacked_root: str = "/kaggle/input/datasets/sealeopard/attacked-pth-files"   
    weights_dir: str = "/kaggle/input/datasets/sealeopard/resnet-pth"            
    out_root: str = "/kaggle/working"                         
    overlap_list: List[int]   = field(default_factory=lambda: [10,20,30,40,50,60,70,80,90])
    magnitude_list: List[float] = field(default_factory=lambda:
                                        [0.0, 0.01, 0.005, 0.002, 0.0014, 0.001, 0.0005])

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

def num_classes_of(ds): return 100 if ds == "cifar100" else 10

cfg = Config()
print(cfg)

Config(ds_name='cifar10', net_type='resnet', adv_type='FGSM', batch_size=100, attacked_root='/kaggle/input/datasets/sealeopard/attacked-pth-files', weights_dir='/kaggle/input/datasets/sealeopard/resnet-pth', out_root='/kaggle/working', overlap_list=[10, 20, 30, 40, 50, 60, 70, 80, 90], magnitude_list=[0.0, 0.01, 0.005, 0.002, 0.0014, 0.001, 0.0005])


In [ ]:
def load_model(cfg):
    nc = num_classes_of(cfg.ds_name)
    model = models.ResNet34(num_c=nc)
    ckpt = os.path.join(cfg.weights_dir, f"{cfg.net_type}_{cfg.ds_name}.pth")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.to(DEVICE).eval()
    return model, nc

def get_train_loader(cfg):
    tf = T.Compose([T.ToTensor(), T.Normalize(CIFAR_MEAN, CIFAR_STD)])
    train_loader, _ = data_loader.getTargetDataSet(
        cfg.ds_name, cfg.batch_size, tf, "/kaggle/working/data")
    return train_loader

def feature_dims(model):
    dummy = torch.rand(2, 3, 32, 32, device=DEVICE)
    with torch.no_grad():
        _, outs = model.feature_list(dummy)
    return [o.size(1) for o in outs]        

def load_attacked(cfg):
    d = os.path.join(cfg.attacked_root, cfg.ds_name.upper(), cfg.adv_type)
    tag = f"{cfg.net_type}_{cfg.ds_name}_{cfg.adv_type}"
    load = lambda p: torch.load(os.path.join(d, p), map_location="cpu")
    clean = load(f"clean_data_{tag}.pth")
    adv   = load(f"adv_data_{tag}.pth")
    noisy = load(f"noisy_data_{tag}.pth")
    n = cfg.batch_size * (len(clean) // cfg.batch_size)  
    return clean[:n], adv[:n], noisy[:n]

_STD_T = torch.tensor(CIFAR_STD, device=DEVICE).view(1, 3, 1, 1)

In [ ]:
def sample_estimator(model, num_classes, dims, train_loader):
    n_layers = len(dims)
    feats = [[] for _ in range(n_layers)]
    labels = []
    with torch.no_grad():
        for data, target in train_loader:
            data = data.to(DEVICE)
            _, outs = model.feature_list(data)
            for k in range(n_layers):
                f = outs[k].view(outs[k].size(0), outs[k].size(1), -1).mean(2)
                feats[k].append(f.cpu())
            labels.append(target.cpu())
    labels = torch.cat(labels)
    feats = [torch.cat(fl, 0) for fl in feats]

    sample_mean, precision = [], []
    lasso = sklearn.covariance.EmpiricalCovariance(assume_centered=False)
    for k in range(n_layers):
        Fk = feats[k]
        means = torch.stack([Fk[labels == c].mean(0) for c in range(num_classes)]) 
        centered = Fk - means[labels]
        lasso.fit(centered.numpy())
        sample_mean.append(means.to(DEVICE))
        precision.append(torch.from_numpy(lasso.precision_).float().to(DEVICE))
    return sample_mean, precision

def _gaussian_scores(feat, mean_c, prec):
    cols = []
    for c in range(mean_c.size(0)):
        zf = feat - mean_c[c]
        cols.append(-0.5 * ((zf @ prec) * zf).sum(1))
    return torch.stack(cols, 1)

def maha_layer_scores(model, data_tensor, num_classes, sample_mean, precision,
                      layer_index, magnitude, batch_size=100):
    model.eval()
    out = []
    n = data_tensor.size(0)
    for b in range(n // batch_size):
        data = data_tensor[b*batch_size:(b+1)*batch_size].to(DEVICE).clone().detach()
        data.requires_grad_(True)
        feat = model.intermediate_forward(data, layer_index)
        feat = feat.view(feat.size(0), feat.size(1), -1).mean(2)
        g = _gaussian_scores(feat, sample_mean[layer_index], precision[layer_index])
        pred = g.argmax(1)
        zf = feat - sample_mean[layer_index][pred]
        pure = -0.5 * ((zf @ precision[layer_index]) * zf).sum(1)
        loss = (-pure).mean()
        loss.backward()

        grad = (data.grad.detach() >= 0).float() * 2 - 1    
        grad = grad / _STD_T
        temp = data.detach() - magnitude * grad
        with torch.no_grad():
            nf = model.intermediate_forward(temp, layer_index)
            nf = nf.view(nf.size(0), nf.size(1), -1).mean(2)
            ng = _gaussian_scores(nf, sample_mean[layer_index], precision[layer_index])
            out.append(ng.max(1).values.cpu())
    return torch.cat(out).numpy()

def maha_all_layers(model, data_tensor, num_classes, sample_mean, precision,
                    dims, magnitude, batch_size=100):
    cols = [maha_layer_scores(model, data_tensor, num_classes, sample_mean, precision,
                              li, magnitude, batch_size) for li in range(len(dims))]
    return np.stack(cols, 1)                              

In [ ]:
def mle_batch(data, batch, k):
    data = np.asarray(data, dtype=np.float32)
    batch = np.asarray(batch, dtype=np.float32)
    k = min(k, len(data) - 1)
    f = lambda v: -k / np.sum(np.log(v / v[-1]))
    a = cdist(batch, data)
    a = np.sort(a, axis=1)[:, 1:k + 1]
    return np.apply_along_axis(f, axis=1, arr=a)

def lid_all_overlaps(model, clean_t, adv_t, noisy_t, dims, overlaps, batch_size=100):
    n_layers = len(dims)
    acc = {k: {"clean": [], "adv": [], "noisy": []} for k in overlaps}
    n = clean_t.size(0)
    for b in range(n // batch_size):
        sl = slice(b*batch_size, (b+1)*batch_size)
        def layer_feats(t):
            with torch.no_grad():
                _, outs = model.feature_list(t[sl].to(DEVICE))
            return [o.view(o.size(0), o.size(1), -1).mean(2).cpu().numpy() for o in outs]
        Fc, Fa, Fn = layer_feats(clean_t), layer_feats(adv_t), layer_feats(noisy_t)
        for k in overlaps:
            cc = [mle_batch(Fc[j], Fc[j], k).reshape(-1, 1) for j in range(n_layers)]
            ca = [mle_batch(Fc[j], Fa[j], k).reshape(-1, 1) for j in range(n_layers)]
            cn = [mle_batch(Fc[j], Fn[j], k).reshape(-1, 1) for j in range(n_layers)]
            acc[k]["clean"].append(np.concatenate(cc, 1))
            acc[k]["adv"].append(np.concatenate(ca, 1))
            acc[k]["noisy"].append(np.concatenate(cn, 1))
    out = {}
    for k in overlaps:
        out[k] = (np.concatenate(acc[k]["clean"]),
                  np.concatenate(acc[k]["adv"]),
                  np.concatenate(acc[k]["noisy"]))
    return out

In [ ]:
def stack_with_labels(adv_feats, clean_feats, noisy_feats):
    pos = np.concatenate([clean_feats, noisy_feats], 0)
    X = np.concatenate([adv_feats, pos], 0).astype(np.float32)
    y = np.concatenate([np.zeros(len(adv_feats)), np.ones(len(pos))]).reshape(-1, 1)
    return np.concatenate([X, y.astype(np.float32)], 1)

def extract_and_save(cfg, verbose=True):
    model, nc = load_model(cfg)
    dims = feature_dims(model)
    train_loader = get_train_loader(cfg)
    clean_t, adv_t, noisy_t = load_attacked(cfg)
    out_dir = os.path.join(cfg.out_root, f"{cfg.net_type}_{cfg.ds_name}", cfg.adv_type)
    os.makedirs(out_dir, exist_ok=True)
    if verbose:
        print(f"[{cfg.ds_name}/{cfg.adv_type}] layers={len(dims)} dims={dims} "
              f"n={clean_t.size(0)} -> {out_dir}")

    lid = lid_all_overlaps(model, clean_t, adv_t, noisy_t, dims,
                           cfg.overlap_list, cfg.batch_size)
    for k in cfg.overlap_list:
        c, a, n = lid[k]
        arr = stack_with_labels(a, c, n)
        np.save(os.path.join(out_dir, f"LID_{k}_{cfg.ds_name}_{cfg.adv_type}.npy"), arr)
    if verbose: print(f"  saved {len(cfg.overlap_list)} LID files")

    sample_mean, precision = sample_estimator(model, nc, dims, train_loader)
    for m in cfg.magnitude_list:
        Mc = maha_all_layers(model, clean_t, nc, sample_mean, precision, dims, m, cfg.batch_size)
        Ma = maha_all_layers(model, adv_t,   nc, sample_mean, precision, dims, m, cfg.batch_size)
        Mn = maha_all_layers(model, noisy_t, nc, sample_mean, precision, dims, m, cfg.batch_size)
        arr = stack_with_labels(Ma, Mc, Mn)
        np.save(os.path.join(out_dir, f"Mahalanobis_{m}_{cfg.ds_name}_{cfg.adv_type}.npy"), arr)
    if verbose: print(f"  saved {len(cfg.magnitude_list)} Mahalanobis files")
    return out_dir

In [7]:
# --- Run for the configured attack --------------------------------------
#out_dir = extract_and_save(cfg)

In [ ]:
def sanity(out_dir, cfg):
    def quick_auroc(path):
        d = np.load(path)
        X, y = d[:, :-1], d[:, -1]        # y: adv=0, clean/noisy=1
        auc = roc_auc_score(y, X.sum(1))
        return max(auc, 1 - auc)         
    for pat in ["Mahalanobis", "LID"]:
        files = sorted(glob.glob(os.path.join(
            out_dir, f"{pat}_*_{cfg.ds_name}_{cfg.adv_type}.npy")))
        best = max((quick_auroc(f), os.path.basename(f)) for f in files)
        print(f"  {pat:12s} best smoke-AUROC: {best[0]*100:5.4f}%  ({best[1]})")

In [ ]:
import glob
for adv_type in ["FGSM", "BIM", "DeepFool", "CWL2"]:
    cfg.adv_type = adv_type
    out_dir = extract_and_save(cfg)   
    sanity(out_dir, cfg) 

100%|██████████| 170M/170M [36:04<00:00, 78.8kB/s]


[cifar10/FGSM] layers=5 dims=[64, 64, 128, 256, 512] n=6700 -> /kaggle/working/resnet_cifar10/FGSM
  saved 9 LID files
  saved 7 Mahalanobis files
  Mahalanobis  best smoke-AUROC: 99.82%  (Mahalanobis_0.002_cifar10_FGSM.npy)
  LID          best smoke-AUROC: 95.77%  (LID_60_cifar10_FGSM.npy)
[cifar10/BIM] layers=5 dims=[64, 64, 128, 256, 512] n=9100 -> /kaggle/working/resnet_cifar10/BIM
  saved 9 LID files
  saved 7 Mahalanobis files
  Mahalanobis  best smoke-AUROC: 98.74%  (Mahalanobis_0.002_cifar10_BIM.npy)
  LID          best smoke-AUROC: 90.37%  (LID_80_cifar10_BIM.npy)
[cifar10/DeepFool] layers=5 dims=[64, 64, 128, 256, 512] n=9000 -> /kaggle/working/resnet_cifar10/DeepFool
  saved 9 LID files
  saved 7 Mahalanobis files
  Mahalanobis  best smoke-AUROC: 90.20%  (Mahalanobis_0.0005_cifar10_DeepFool.npy)
  LID          best smoke-AUROC: 76.79%  (LID_10_cifar10_DeepFool.npy)
[cifar10/CWL2] layers=5 dims=[64, 64, 128, 256, 512] n=8500 -> /kaggle/working/resnet_cifar10/CWL2
  saved 9 LI